# Simple Morphological Tree Examples

This notebook introduces the basic `mmcfilters` workflow on a small synthetic image and then repeats the same ideas on a real coin image.


## 1. Prepare the environment

Prepare the notebook environment outside the notebook as described in `notebooks/README.md`. The next cell imports the installed package directly with `import mmcfilters`; it performs no local installation or build-tree loading.


In [ ]:
import mmcfilters


## 2. Import the library


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv
import mmcfilters

def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)
from pathlib import Path
import mtviz as viz
if not hasattr(viz, "show_level_sets"):
    viz.show_level_sets = getattr(viz, "showLevelSets", lambda *args, **kwargs: None)
from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
output_notebook()


def create_component_tree(image, is_maxtree, radius=1.5):
    if is_maxtree:
        return mmcfilters.MorphologicalTreeFactory.createMaxTree(image, radius=radius)
    return mmcfilters.MorphologicalTreeFactory.createMinTree(image, radius=radius)


def print_tree_with_attribute(attribute_type, attribute_by_node):
    return lambda tree, node_id: f"id:{node_id}, {attribute_type.name}: {attribute_by_node[node_id]}"


def show_component_tree(tree, image=None, label=None):
    label = label or (lambda tree, node_id: f"id:{node_id}")

    def walk(node_id, depth=0):
        print("  " * depth + label(tree, node_id))
        for child_id in tree.getChildren(node_id):
            walk(child_id, depth + 1)

    if image is not None:
        plt.figure(figsize=(5, 5))
        plt.imshow(image, cmap='gray', vmax=255, vmin=0)
        plt.axis('off')
        plt.show()

    walk(tree.getRoot())


def show_tree(tree, label=None):
    show_component_tree(tree, label=label)


def getPlotTree(tree, title):
    show_component_tree(tree)
    return None


## 3. Create a morphological tree from an input image

A component tree represents connected level sets as nodes. In this example, the synthetic image is small enough that the tree structure can be printed and compared with the pixel values.


In [ ]:
#input_image = load_grayscale("../dat/imgTeste.png")

input_image = np.array([
        [203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203],
        [203,203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203,203],
        [203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203],
        [203,203, 78, 78,126,126,126,126,126,126,126, 78, 78, 78, 78, 78, 78, 78,203,203,203, 54, 54,203,203],
        [203,203, 78, 78,126, 38, 38, 38,126,126,126, 78, 78, 78, 78, 78, 78, 78,203,203, 54, 54, 54, 54,203],
        [203,203, 78, 78,126, 38, 38, 38,126,126,126, 78, 78, 78, 78, 78, 78, 78,203, 54, 54, 54, 80, 54,203],
        [203,203, 78, 78,126, 38, 38, 38,126, 78, 78, 78, 78, 78, 78, 78, 78, 78,203, 54, 80, 54, 54, 54,203],
        [203, 78, 78, 78,126, 38, 38,126,126, 78, 78, 78,203,203,203,203,203,203,203, 54, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126,126, 78, 78,203,203,203,203,203,203,203,203, 54, 54, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203,203, 54, 80, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203, 54, 54, 54, 54, 54,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203,203, 54, 54, 54,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,203,203,203,203,203,203,203,203,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,203,126,126,126,126,203,203,203,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,126,126,126,126,126,126,126,126,126,203,203,203],
        [203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,126,126,126,126,126,126, 72,126,126,203,203,203],
        [203, 78, 78, 78, 78, 78, 78,161,161,161, 78, 78,203,126,126,126,126,126,126, 72, 72,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78,203,126,126,126,126,126, 72, 72, 72,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 30, 30,161, 78,203,203,126,126,126, 72, 72, 72, 72, 72,126,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 90, 30,161, 78, 78,203,126,126, 72, 72, 72, 72, 72, 72, 72,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 30, 30,161, 78, 78,203,203,126,126,126,126,126,126,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78, 78, 78,203,203,126,126,126,126,126,126,126,203,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78, 78, 78,203,203,203,203,126,126,126,126,203,203,203],
        [203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203,203,203,203],
        [203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203]
], dtype=np.uint8)
input_image = np.ascontiguousarray(input_image, dtype=np.uint8)
(num_rows, num_cols) = input_image.shape

is_max_tree = False
tree = create_component_tree(input_image, is_max_tree)
viz.show_level_sets(input_image)


In [ ]:
show_component_tree(tree, image=input_image)

## 4. Available attributes

Attributes summarize geometric, radiometric, or topological properties of each node. Listing the available attributes is a good first step before choosing a filtering criterion.


In [ ]:
attribute_enum = type(mmcfilters.Attribute.AREA)
describe = {
    attribute_name: mmcfilters.Attribute.describe(attribute_value)
    for attribute_name in dir(mmcfilters.Attribute)
    if attribute_name.isupper()
    for attribute_value in [getattr(mmcfilters.Attribute, attribute_name)]
    if isinstance(attribute_value, attribute_enum)
}

df = pd.DataFrame(describe.items(), columns=['Attribute type', 'Description'])
df.style.set_caption("<H3><b>Available attributes</b></H3>")

## 5. Compute a single attribute

`computeSingleAttribute` returns one value per tree node. The gray-height attribute is used here because it is easy to inspect on the printed tree.


In [ ]:
attribute_type = mmcfilters.Attribute.GRAY_HEIGHT
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, attribute_type)

print(f"{attribute_type.name} (numpy): {attribute_by_node}")

In [ ]:
show_tree(tree, print_tree_with_attribute(attribute_type, attribute_by_node))

## 6. Compute multiple attributes

`computeAttributes` evaluates several attributes together and returns a node-by-attribute table. This is useful when a filter or analysis combines multiple criteria.


In [ ]:
attribute_indices, attribute_matrix = mmcfilters.Attribute.computeAttributes(tree, [mmcfilters.Attribute.AREA, mmcfilters.Attribute.GRAY_HEIGHT, mmcfilters.Attribute.VOLUME, mmcfilters.Attribute.RELATIVE_VOLUME])
#print(attribute_matrix[:, attribute_indices['GRAY_HEIGHT']])
pd.DataFrame(attribute_matrix, columns=attribute_indices, index=pd.Index(range(len(attribute_matrix)), name="NodeID"))

## 7. Attribute filtering

Attribute filters remove or preserve nodes according to a criterion and then reconstruct an image from the modified tree. This example uses the subtractive rule with the gray-height values.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, mmcfilters.Attribute.GRAY_HEIGHT)
threshold = 80

attribute_filter = mmcfilters.AttributeFilters(tree)
filtered_image = attribute_filter.filteringSubtractiveRule(attribute_by_node > threshold)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter')

## 8. Extract extinction values

Extinction values rank extrema by the importance of the attribute that disappears when components merge. They are often more stable than a fixed threshold on raw attribute values.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, mmcfilters.Attribute.GRAY_HEIGHT) # the attribute must be increasing

In [ ]:
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node)
for leaf_id, cutoff_node_id, extinction in extinction_values.getRegionalExtrema():
    print("Regional extremum (leaf):", leaf_id)
    print("Extinction value: ", extinction)
    print("Persistence node where the regional extremum still exists:", cutoff_node_id)
    print("The regional extremum disappears at:", tree.getNodeParent(cutoff_node_id))
    print()

show_component_tree(tree, image=input_image)


## 9. Filter by extinction values

Here the filter keeps only the most relevant extrema according to the extinction ranking. This provides a compact way to control the number of preserved structures.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, mmcfilters.Attribute.GRAY_HEIGHT)
attribute_filter = mmcfilters.AttributeFilters(tree)

num_leaves_to_keep = 3 # keep num_leaves_to_keep leaves with the highest extinction values
selection_policy = mmcfilters.ExtinctionSelectionPolicy.byTopK(num_leaves_to_keep)
filtered_image = attribute_filter.filteringByExtinction(attribute_by_node, selection_policy)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 3 regional extrema')

## 10. Work with the coin image

The same workflow is applied to a real image: direct attribute filtering, extinction-value filtering, and saliency-map construction.


In [ ]:
# 1. Filtering

from skimage import data, img_as_float
input_image = np.ascontiguousarray(data.coins(), dtype=np.uint8)
(num_rows, num_cols) = input_image.shape

is_max_tree = True
tree = create_component_tree(input_image, is_max_tree)

attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)
threshold = 500

attribute_filter = mmcfilters.AttributeFilters(tree)
filtered_image = attribute_filter.filteringSubtractiveRule(attribute_by_node > threshold)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter')

In [ ]:
# 2. Filtering by extinction values

input_image = filtered_image

is_max_tree = True
tree = create_component_tree(input_image, is_max_tree)
attribute_filter = mmcfilters.AttributeFilters(tree)
attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)

num_leaves_to_keep = 6 # keep num_leaves_to_keep leaves with the highest extinction values
filtered_image_6 = attribute_filter.filteringByExtinction(attribute_by_node, mmcfilters.ExtinctionSelectionPolicy.byTopK(num_leaves_to_keep))

num_leaves_to_keep = 24 # keep num_leaves_to_keep leaves with the highest extinction values
filtered_image_24 = attribute_filter.filteringByExtinction(attribute_by_node, mmcfilters.ExtinctionSelectionPolicy.byTopK(num_leaves_to_keep))

plt.figure(figsize=(15, 5))
plt.subplot(1,3, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,3, 2)
plt.imshow(filtered_image_6, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 6 regional extrema')

plt.subplot(1,3,3)
plt.imshow(filtered_image_24, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 24 regional extrema')

In [ ]:
# 3. Creating a saliency map

attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)
circularity = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.CIRCULARITY)
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node) # the attribute must be increasing
num_leaves_to_keep = int(tree.numLeafNodes * 1) # 5% of the regional extrema
contours = mmcfilters.ContourComputation.extraction(tree)

contour_image = np.zeros((num_rows*num_cols), dtype=np.float32)
importance = num_leaves_to_keep
for leaf_id, cutoff_node_id, extinction in extinction_values.getRegionalExtrema()[:num_leaves_to_keep]:
    for p in contours.getContour(cutoff_node_id):
        contour_image[p]= circularity[cutoff_node_id]
        #contour_image[p]= importance
    importance -= 1


plt.figure(figsize=(5, 5))
plt.imshow(contour_image.reshape(num_rows, num_cols), cmap='Grays')
plt.axis('off')
plt.show()


## 11. Combine shape criteria on contours

This final section uses area and circularity to build contour-based saliency maps. The approach is limited to increasing attributes, so the chosen criteria should be checked before reuse.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)
circularity = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.CIRCULARITY)
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node) # the attribute must be increasing
num_leaves_to_keep = int(tree.numLeafNodes * 1)

contours = mmcfilters.ContourComputation.extraction(tree)

contour_image = np.zeros((num_rows*num_cols), dtype=np.float32)
importance = num_leaves_to_keep
for leaf_id, cutoff_node_id, extinction in extinction_values.getRegionalExtrema()[:num_leaves_to_keep]:
    for p in contours.getContour(cutoff_node_id):
        contour_image[p]=circularity[cutoff_node_id]
    importance -= 1



plt.figure(figsize=(15, 5))
plt.subplot(1,2, 1)
plt.imshow(contour_image.reshape(num_rows, num_cols), cmap='Grays')
plt.axis('off')
plt.title('saliency map: importance is the circularity')

plt.subplot(1,2, 2)
saliency_map = extinction_values.contourMap(mmcfilters.ExtinctionSelectionPolicy.byTopK(num_leaves_to_keep), mmcfilters.ExtinctionContourScorePolicy.RankScore)
plt.imshow(saliency_map, cmap='Grays')
plt.axis('off')
plt.title('saliency map: importance is the extinction values')
plt.show()


In [ ]:
# Alternative 1: circularity persistence directly on the tree
# Union-find sweeps the upper-level sets of C without materializing
# a second tree. Topologically, it computes the same 0D persistence.

PERSISTENCE_TOP_K = 20
circularity = np.ascontiguousarray(
    mmcfilters.Attribute.computeSingleTopologyAttribute(
        tree, mmcfilters.Attribute.CIRCULARITY
    ),
    dtype=np.float32,
)

num_node_slots = int(tree.numInternalNodeSlots)
alive_nodes = np.asarray(tree.aliveNodeIds, dtype=np.int64)
node_neighbors = [[] for _ in range(num_node_slots)]
for node_id in alive_nodes:
    node_id = int(node_id)
    parent_id = int(tree.getNodeParent(node_id))
    if parent_id != node_id:
        node_neighbors[node_id].append(parent_id)
        node_neighbors[parent_id].append(node_id)

uf_parent = np.arange(num_node_slots, dtype=np.int64)
active = np.zeros(num_node_slots, dtype=bool)
peak_node = np.full(num_node_slots, -1, dtype=np.int64)
birth_level = np.full(num_node_slots, -np.inf, dtype=np.float64)

def find_root(node_id):
    node_id = int(node_id)
    while uf_parent[node_id] != node_id:
        uf_parent[node_id] = uf_parent[uf_parent[node_id]]
        node_id = int(uf_parent[node_id])
    return node_id

def join_plateau(first_id, second_id):
    first_root = find_root(first_id)
    second_root = find_root(second_id)
    if first_root == second_root:
        return first_root
    survivor = min(first_root, second_root)
    absorbed = max(first_root, second_root)
    uf_parent[absorbed] = survivor
    return survivor

persistence_records = []
processing_order = alive_nodes[
    np.argsort(-circularity[alive_nodes], kind="stable")
]
start = 0
while start < processing_order.size:
    level = float(circularity[int(processing_order[start])])
    stop = start + 1
    while (
        stop < processing_order.size
        and float(circularity[int(processing_order[stop])]) == level
    ):
        stop += 1
    current_nodes = processing_order[start:stop]
    active[current_nodes] = True

    # Adjacent equal-valued nodes form one plateau maximum.
    for node_id in current_nodes:
        node_id = int(node_id)
        for neighbor_id in node_neighbors[node_id]:
            if (
                active[neighbor_id]
                and float(circularity[neighbor_id]) == level
            ):
                join_plateau(node_id, neighbor_id)

    plateau_members = {}
    for node_id in current_nodes:
        plateau_members.setdefault(find_root(node_id), []).append(int(node_id))

    for plateau_root, members in plateau_members.items():
        higher_roots = {
            find_root(neighbor_id)
            for node_id in members
            for neighbor_id in node_neighbors[node_id]
            if (
                active[neighbor_id]
                and float(circularity[neighbor_id]) > level
            )
        }
        higher_roots.discard(find_root(plateau_root))
        if not higher_roots:
            peak_node[plateau_root] = min(members)
            birth_level[plateau_root] = level
            continue

        # Elder rule: the maximum with the highest birth survives.
        survivor = min(
            higher_roots,
            key=lambda root_id: (-birth_level[root_id], peak_node[root_id]),
        )
        saddle_node = min(members)
        for dying_root in sorted(higher_roots - {survivor}):
            persistence_records.append({
                "peakNode": int(peak_node[dying_root]),
                "saddleNode": saddle_node,
                "birthCircularity": float(birth_level[dying_root]),
                "deathCircularity": level,
                "persistence": float(birth_level[dying_root] - level),
                "dominant": False,
            })
            uf_parent[dying_root] = survivor
        uf_parent[find_root(plateau_root)] = survivor
    start = stop

# The global maximum is dominant and receives no finite death.
final_root = find_root(int(tree.getRoot()))
persistence_records.append({
    "peakNode": int(peak_node[final_root]),
    "saddleNode": int(tree.getRoot()),
    "birthCircularity": float(birth_level[final_root]),
    "deathCircularity": np.nan,
    "persistence": np.inf,
    "dominant": True,
})
finite_persistence = sorted(
    (record for record in persistence_records if not record["dominant"]),
    key=lambda record: (-record["persistence"], record["peakNode"]),
)
selected_persistence = finite_persistence[:PERSISTENCE_TOP_K]

node_persistence = np.zeros(num_node_slots, dtype=np.float32)
for record in selected_persistence:
    node_persistence[record["peakNode"]] = record["persistence"]

incremental_contours = (
    mmcfilters.HierarchySaliencyMapProjection.computeIncrementalNodeContours(tree)
)
persistence_edge_map = (
    mmcfilters.HierarchySaliencyMapProjection.projectNodeValuation(
        incremental_contours, node_persistence
    )
)
persistence_image = (
    mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
        persistence_edge_map
    )
)

display(pd.DataFrame(selected_persistence).head(10))
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
axes[0].imshow(input_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Filtered image")
artist = axes[1].imshow(persistence_image, cmap="magma", vmin=0)
axes[1].set_title("Circularity persistence on contours")
for axis in axes:
    axis.axis("off")
fig.colorbar(artist, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()
print(f"finite maxima: {len(finite_persistence)}")
print(f"displayed maxima: {len(selected_persistence)}")


In [ ]:
# Alternative 2: area extinction + circularity, followed by increasing ranks
# Circularity selects shapes; integer levels define the hierarchy.

MIN_AREA_EXTINCTION = 1200.0
MIN_CIRCULARITY = 0.80

area = np.ascontiguousarray(
    mmcfilters.Attribute.computeSingleTopologyAttribute(
        tree, mmcfilters.Attribute.AREA
    ),
    dtype=np.float32,
)
circularity = np.ascontiguousarray(
    mmcfilters.Attribute.computeSingleTopologyAttribute(
        tree, mmcfilters.Attribute.CIRCULARITY
    ),
    dtype=np.float32,
)
area_extinctions = mmcfilters.ExtinctionValues(tree, area)
float32_max = np.finfo(np.float32).max
finite_area_records = [
    record
    for record in area_extinctions.getRegionalExtrema()
    if float(record[2]) < float32_max
]
selected_area_circularity = [
    record
    for record in finite_area_records
    if (
        float(record[2]) >= MIN_AREA_EXTINCTION
        and float(circularity[int(record[1])]) >= MIN_CIRCULARITY
    )
]

# Selected cutoff nodes and the root are retained.
# Post-order propagation is equivalent to contracting the other levels.
retained_nodes = np.zeros(tree.numInternalNodeSlots, dtype=bool)
retained_nodes[int(tree.getRoot())] = True
for _leaf_id, cutoff_node_id, _extinction in selected_area_circularity:
    retained_nodes[int(cutoff_node_id)] = True

hierarchy_rank = np.zeros(tree.numInternalNodeSlots, dtype=np.int32)
for node_id in tree.getPostOrderNodes():
    node_id = int(node_id)
    child_ranks = [
        hierarchy_rank[int(child_id)]
        for child_id in tree.getChildren(node_id)
    ]
    hierarchy_rank[node_id] = (
        max(child_ranks, default=0) + int(retained_nodes[node_id])
    )

mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(
    tree, hierarchy_rank, nonnegative=True
)
filtered_hierarchy_edge_map = (
    mmcfilters.HierarchySaliencyMap.computeSaliencyEdgeMap(
        tree, hierarchy_rank
    )
)
filtered_hierarchy_image = (
    mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
        filtered_hierarchy_edge_map
    )
)

selection_rows = [
    {
        "leaf": int(leaf_id),
        "cutoffNode": int(cutoff_node_id),
        "areaExtinction": float(extinction),
        "circularity": float(circularity[int(cutoff_node_id)]),
    }
    for leaf_id, cutoff_node_id, extinction in selected_area_circularity
]
selection_frame = pd.DataFrame(
    selection_rows,
    columns=["leaf", "cutoffNode", "areaExtinction", "circularity"],
)
display(selection_frame.sort_values(
    "areaExtinction", ascending=False
).head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
axes[0].imshow(input_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Filtered image")
artist = axes[1].imshow(filtered_hierarchy_image, cmap="magma", vmin=0)
axes[1].set_title("LCA saliency of the filtered hierarchy")
for axis in axes:
    axis.axis("off")
fig.colorbar(artist, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()
print(f"selected extrema: {len(selected_area_circularity)}")
print(f"hierarchy levels: {int(hierarchy_rank.max()) + 1}")
